## (1) Importing modules

In [ ]:
#!/usr/bin/python
# -*- coding: UTF-8 -*-

#__modification time__ = 2026-05-10
#__author__ = Qi Zhou, Helmholtz Centre Potsdam - GFZ German Research Centre for Geosciences
#__find me__ = qi.zhou@gfz.de, qi.zhou.geo@gmail.com, https://github.com/Qi-Zhou-Geo
# Please do not distribute this code without the author's permission

import numpy as np
import pandas as pd

from obspy import read
from obspy.core import UTCDateTime
from obspy.clients.fdsn import Client

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.gridspec as gridspec


from tqdm import tqdm

# region <add the sys.path to search for custom modules>
import sys
from pathlib import Path

current_dir = Path.cwd()
project_root = current_dir.parent.parent
sys.path.append(str(project_root))


print(f"Current dir: {current_dir}\n"
      f"Project root: {project_root}")
# endregion


# import the custom functions
from calculate_features.s1_cal_TypeA_TypeB import cal_attributes_A, cal_attributes_B

## (1) Load the pre-downloaded data

In [ ]:
st = read(f"{current_dir}/data/cooked_2014-07-12to2014-07-13.mseed")

# double check the meta data
print(st[0].stats)

## (2) Calculate the seismic features
### (2-1) what features will we get?

In [ ]:
# check the paper for details of A features: https://doi.org/10.1029/2024JF007691
feature_Name_A = ['time_window_start', 'time_stamps', 'station', 'component',
                  'digit1','digit2','digit3','digit4','digit5', 'digit6','digit7','digit8','digit9',
                  'max', 'goodness', 'iqr', 'magnitude_range', 'alpha', 'ks', 'MannWhitneU', 'follow'] # Benford's Law features

# check the paper for details of B features: https://doi.org/10.1029/2020gl090874, https://doi.org/10.1002/2016gl070709
feature_Name_B = ['time_window_start', 'time_stamps', 'station', 'component',
                  'RappMaxMean', 'RappMaxMedian', 'AsDec', 'KurtoSig','KurtoEnv', 'SkewnessSig','SkewnessEnv',
                  'CorPeakNumber', 'INT1', 'INT2', 'INT_RATIO', 'ES_0', 'ES_1', 'ES_2', 'ES_3', 'ES_4', 'KurtoF_0',
                  'KurtoF_1', 'KurtoF_2', 'KurtoF_3', 'KurtoF_4', 'DistDecAmpEnv','env_max_to_duration', 'RMS', 'IQR', 'MeanFFT', 'MaxFFT',
                  'FmaxFFT', 'FCentroid', 'Fquart1', 'Fquart3', 'MedianFFT', 'VarFFT', 'NpeakFFT', 'MeanPeaksFFT', 'E1FFT','E2FFT','E3FFT', 'E4FFT',
                  'gamma1', 'gamma2', 'gammas', 'SpecKurtoMaxEnv', 'SpecKurtoMedianEnv', 'RatioEnvSpecMaxMean', 'RatioEnvSpecMaxMedian','DistMaxMean',
                  'DistMaxMedian', 'NbrPeakMax', 'NbrPeakMean', 'NbrPeakMedian','RatioNbrPeakMaxMean', 'RatioNbrPeakMaxMedian',
                  'NbrPeakFreqCenter', 'NbrPeakFreqMax', 'RatioNbrFreqPeaks','DistQ2Q1', 'DistQ3Q2', 'DistQ3Q1'] # Waveform, Spectral, and Spectrogram features

### (2-2) Define the parameters for calculating features

In [ ]:
# columns array to store the seismic data-60s for network features
sps = int(st[0].stats.sampling_rate)

input_year = st[0].stats.starttime.year
julday = st[0].stats.starttime.julday

input_station = st[0].stats.station
input_component = st[0].stats.channel

input_window_size = 60 # unit is second

total_seconds = float(st[0].stats.endtime - st[0].stats.starttime)
num_time_step = int(total_seconds / input_window_size) # e.g., 1440 = (24h * 60 minutes) / 1 minute

d0 = UTCDateTime(year=input_year, julday=julday)  # the start day, e.g.2014-07-12T00:00:00


### (2-3) Try the function

In [ ]:
tr = st.copy()
tr.trim(starttime=UTCDateTime("2014-07-12T14:59:00"), endtime=UTCDateTime("2014-07-12T15:01:00"), nearest_sample=False)
seismic_data = tr[0].data
print(seismic_data.shape)


type_A_arr = cal_attributes_A(data_array=seismic_data)
type_B_arr = cal_attributes_B(data_array=seismic_data, sps=sps)

# note: we do not add metadata for the calculated feature array
print(type_A_arr.shape, len(feature_Name_A))
print(type_B_arr.shape, len(feature_Name_B))

### (2-4) Run the loop

In [ ]:
arr_A_list = np.empty(shape=(num_time_step, len(feature_Name_A)), dtype="object")
arr_B_list = np.empty(shape=(num_time_step, len(feature_Name_B)), dtype="object")

for step in tqdm(range(num_time_step), total=num_time_step, desc="Runing calBL_feature"):
    
    d1 = d0 + (step) * input_window_size      # from minute 0, 1, 2, 3, 4
    d2 = d0 + (step + 1) * input_window_size  # from minute 1, 2, 3, 4, 5
    d1_str = UTCDateTime(d1).strftime("%Y-%m-%dT%H:%M:%S")

    # metadata for calculated seismic features
    meta_data = np.array([d1_str, d1.timestamp, input_station, input_component])

    tr = st.copy()
    tr.trim(starttime=d1, endtime=d2, nearest_sample=False)
    seismic_data = tr[0].data[:sps * input_window_size]
    
    type_A_arr = cal_attributes_A(data_array=seismic_data)
    type_B_arr = cal_attributes_B(data_array=seismic_data, sps=sps)

    arr_A = np.append(meta_data, type_A_arr)
    arr_B = np.append(meta_data, type_B_arr)
    
    arr_A_list[step, :] = arr_A
    arr_B_list[step, :] = arr_B
    


## (3) Save the calculated features to local path

In [ ]:
df_A = pd.DataFrame(data=arr_A_list, columns=feature_Name_A)
df_A.to_csv(f"{current_dir}/data/{input_year}_{input_station}_{input_component}_{julday}_A.txt", index=False)

df_B = pd.DataFrame(data=arr_B_list, columns=feature_Name_B)
df_B.to_csv(f"{current_dir}/data/{input_year}_{input_station}_{input_component}_{julday}_B.txt", index=False)

## (4) Visualize the features

In [ ]:
x = np.arange(df_A.shape[0])
start_time = df_A.iloc[0, 0]
feature_name = "goodness"

plt.plot(x, df_A[feature_name].values.astype("float"), color="black")
plt.xlabel(f"Time [UTC+0 from {start_time}]", fontweight='bold')
plt.ylabel(f"{feature_name.capitalize()}", fontweight='bold')
plt.show()

## Task 1: Can you plot the 'ES_X' features?

$$
ES_{f_1-f_2} = \int_{0}^{T} S_{f_1-f_2}(t) \, dt,
$$
where $T$ is the length of the time window, $S_{f_1-f_2}(t)$ is the envelope of seismic signal filtered between frequencies $f_1$ and $f_2$.

In [ ]:
feature_dict = {'ES_0': 'Seismic energy between 0 and 5 Hz', 
                'ES_1': 'Seismic energy between 5 and 10 Hz', 
                'ES_2': 'Seismic energy between 10 and 15 Hz', 
                'ES_3': 'Seismic energy between 15 and 20 Hz', 
                'ES_4': 'Seismic energy between 20 and 25 Hz'}

x = np.arange(df_B.shape[0])
start_time = df_B.iloc[0, 0]

for key, values in feature_dict.items():
    feature_name = key
    plt.plot(x, df_B[feature_name].values.astype("float"), label=feature_name, alpha=0.5)

plt.legend(loc="best", fontsize=6)
plt.xlabel(f"Time [UTC+0 from {start_time}]", fontweight='bold')
plt.ylabel(f"Seismic Energy", fontweight='bold')
plt.show()

## Question 1: What do you think the ML/AI will learn from these featuers?
## tempormal patterns? absolute values? or what else?